# double arm mvp

A runnable walkthrough of both arms of **bind-the-unknown**, built to be shown and talked
through rather than read.

| | Arm 1 · Co-scientist | Arm 2 · Variant ranking |
| --- | --- | --- |
| **Decides** | Which target, and what to measure | Which molecules bind |
| **Primary research object?** | No, it is the allocation layer | **Yes** |
| **Runs today?** | Yes, on public data | No, chemistry represented but unwired |
| **Blocked on** | Nothing | The SynX campaign export |

**The numbering is not a ranking.** Arm 2 is what the project is for. Arm 1 allocates the
scarce experimental capacity Arm 2 consumes.

### How this notebook is built

Two rules, both chosen so the notebook survives being run live in a meeting.

1. **No live database calls.** The ten registered tools read NCBI, UniProt, Europe PMC, EBI
   HMMER, InterPro, PDBe, AlphaFold, RCSB and ChEMBL. Measured cost spans 0.7 s to 94 s per
   call, so a demo wired to them is one outage away from failing in front of an audience.
   Every cell here reads the **committed run records** instead, except the selector, which is
   pure decision logic and needs no network at all.
2. **Nothing is retyped.** Numbers come out of `runs/` at execution time. If a record changes,
   the notebook changes with it, and no figure here can drift from the recorded result.

To run the live pipeline instead, the commands are in the README under *Running it*.


## Outline

Status marks what is already wired in this notebook, not what exists in the repo.

| § | Section | Shows | Status |
| --- | --- | --- | --- |
| 0 | Setup and data availability | Which records this machine can see | **built** |
| 1.1 | Reachability, not relevance | The selector ranking 10 tools from a bare accession | **built** |
| 1.2 | What the co-scientist arm found | The eight recorded experiments, one line each | **built** |
| 1.3 | Genome to ranked targets | Recovered countermeasure targets, 3 viruses + 1 bacterium | to fill |
| 1.4 | The identification cliff | Where the pipeline stops rather than degrades | to fill |
| 1.5 | The surface a macrocycle would bind | Patch medians with spread, and why the gate is borderline | to fill |
| 2.1 | The surviving result | Gain against coverage, from the committed per-pool records | **built** |
| 2.2 | Tested by intervention | Narrow the pool, recover the effect; destroy sharing, remove it | to fill |
| 2.3 | What Arm 2 is blocked on | The null chemistry fields, stated plainly | to fill |
| 3 | What to say about this | The three claims worth defending, and the one caveat | to fill |

### The spine of the story

Arm 1 earns its keep with one idea: **a tool is callable because its inputs exist, not because
it sounds relevant.** From a bare genome accession, 5 of 10 tools are callable and
identification unlocks exactly the other 5, so the dependency graph has a single bottleneck.
§1.1 shows that computed live.

Arm 2 earns its keep with one condition: **sequence-to-binding is learnable from a handful of
labels only when the candidate pool shares sequence coordinates with the labelled ones.** That
is a property of a library's design, computable before any assay runs. §2.1 shows it.

The honest caveat, which belongs in the same breath as the result: every landscape behind it is
canonical residues on a folded protein domain. **No macrocycle and no non-canonical residue has
been measured anywhere in this repository.**


## 0 · Setup and data availability

Each arm reads a different slice of `runs/`, and the two slices have different git status. Arm 1's eight experiment records are **tracked**. Arm 2's per-pool records are **gitignored by design**, so they exist on the machine that ran them and not in a fresh clone. This cell reports what is actually reachable rather than letting a later cell fail.

In [1]:
import os, json, glob, re
from pathlib import Path

# The user matplotlibrc on this machine carries `backend : inline`, which is not a valid
# backend outside IPython and prints a red warning on import. Point matplotlib at its own
# config dir so the demo starts clean. Must happen before matplotlib is imported.
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplcache"))

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hpc_pilots").is_dir())
RUNS = ROOT / "runs"
os.chdir(ROOT)          # so `from hpc_pilots import ...` resolves
print(f"repo root: {ROOT}")

# Arm 1: the tracked experiment records, in the order the README tells the story.
ARM1 = ["intake_validation", "holdout_ladder", "identification_cascade", "cross_kingdom_intake",
        "surface_markers", "surface_in_ranker", "amenability_gate", "coverage_channel"]

# Arm 2: written by `python -m hpc_pilots summarize`, and gitignored.
ARM2 = {"method gain table": RUNS / "summary" / "method_policy_gain.csv",
        "target gain table": RUNS / "summary" / "target_metrics.csv"}

rows = [{"arm": "1", "source": f"runs/{n}.json", "present": (RUNS / f"{n}.json").exists()}
        for n in ARM1]
rows += [{"arm": "1", "source": "runs/target_ranks.csv",
          "present": (RUNS / "target_ranks.csv").exists()}]
rows += [{"arm": "2", "source": str(p.relative_to(ROOT)), "present": p.exists()}
         for p in ARM2.values()]

avail = pd.DataFrame(rows)
missing = avail.loc[~avail.present, "source"].tolist()
print(f"{int(avail.present.sum())}/{len(avail)} sources present")
if missing:
    print("\nmissing (rebuild with `python -m hpc_pilots summarize`):")
    for m in missing:
        print("   ", m)
avail

repo root: /Users/ww/Desktop/Desktop - William’s MacBook Pro/YALE/MATS/bind-the-unknown
11/11 sources present


,arm,source,present
0,1,runs/intake_validation.json,True
1,1,runs/holdout_ladder.json,True
2,1,runs/identification_cascade.json,True
3,1,runs/cross_kingdom_intake.json,True
4,1,runs/surface_markers.json,True
5,1,runs/surface_in_ranker.json,True
6,1,runs/amenability_gate.json,True
7,1,runs/coverage_channel.json,True
8,1,runs/target_ranks.csv,True
9,2,runs/summary/method_policy_gain.csv,True


---
## 1 · The co-scientist arm

### 1.1 · Reachability, not relevance

This is the one idea worth carrying out of Arm 1, and the cell below is the whole argument.

A campaign starts from a single fact: a pathogen genome accession. Ask which tool to call and
the tempting answer is whichever sounds most apposite. That answer is unavailable, because from
a bare accession a literature query **has no target name to query for** and a structural lookup
**has no accession to look up**. Those tools are not expensive and not irrelevant. They are
*unreachable*, and no amount of matching the query against tool summaries discovers it, because
the obstruction is in the state and not in the text. The earlier text-matching retriever was
measured as returning nothing at this scale.

So each tool declares what it needs in the state and what its return puts there, and the
selector orders every tool on a tuple: **callable before blocked; then whichever unblocks more
other tools; then whichever speaks to more unresolved axes; then name**, always, so the order
is total and a run reproduces.

Step 2 before step 3 is the real judgement. A call that widens the action set changes what
every later round may do; a call that speaks to an axis changes one number. Under a budget that
binds, the option value is worth more than the increment. It is also the assumption most likely
to be wrong, so it sits as one comparison in a documented tuple rather than a weight buried in
a sum, leaving a single term to change.

**Blocked tools are reported, never dropped.** A policy shown only its callable moves cannot
tell a tool that is useless from a tool that is one call away, and under a budget the second is
the more useful thing to know.

In [2]:
from hpc_pilots import selector

start = selector.from_sequence()          # a genome accession, and seven open axes
proposals = selector.rank(start)

print(f"state      : {sorted(start.known)}")
print(f"unresolved : {len(start.unresolved)} axes")
print(f"tools      : {len(proposals)} proposed, "
      f"{sum(p.callable_now for p in proposals)} callable now\n")

board = pd.DataFrame([{
    "tool": p.tool,
    "callable": "yes" if p.callable_now else "—",
    "unlocks": len(p.unlocks),
    "axes": len(p.gates_unresolved),
    "blocked on": ", ".join(p.blocked_on) or "—",
} for p in proposals])

first = selector.first(start)
print(f"first call : {first.tool}  ({first.why})")
print(f"it unlocks : {', '.join(first.unlocks)}")
board

state      : ['genome_accession']
unresolved : 7 axes
tools      : 10 proposed, 5 callable now

first call : intake.rank_targets  (callable now, and unblocks bioactivity.prior_binders, families.entries, geometry.surface_markers, structures.folded, structures.predicted)
it unlocks : bioactivity.prior_binders, families.entries, geometry.surface_markers, structures.folded, structures.predicted


,tool,callable,unlocks,axes,blocked on
0,intake.rank_targets,yes,5,0,—
1,structures.evidence,yes,0,1,—
2,chemistry.library_spec,yes,0,0,—
3,literature.search,yes,0,0,—
4,similarity.homologs,yes,0,0,—
5,geometry.surface_markers,—,0,1,protein_accession
6,bioactivity.prior_binders,—,0,0,protein_accession
7,families.entries,—,0,0,protein_accession|protein_sequence
8,structures.folded,—,0,0,protein_sequence
9,structures.predicted,—,0,0,protein_accession


**Read the table top down.** Five tools are callable and five are blocked, every blocked one
naming the fact it lacks. `intake.rank_targets` wins not because identification sounds
important but because it is the only callable tool that unlocks any other, and it unlocks
exactly the five that are blocked. The dependency graph therefore has a **single bottleneck**,
and identification is it. It also happens to be the only tool that enumerates and ranks
candidates rather than answering about one you have already named.

Three callable tools unblock nothing and bear on no unresolved axis, and the selector says so
in as many words rather than quietly ranking them last.

**What it deliberately does not do: rank by cost.** Seven of the ten specs carry measured wall
time; three still declare zero, and for the deposited-structure tool that zero is wrong by
about 60 seconds and 36 outbound requests. A cost-weighted order would be a preference computed
partly from numbers nobody has taken, so cost waits in the tie-break slot until the record is
complete.

### 1.2 · What the co-scientist arm found

Eight recorded experiments, each carrying its own question and answer. Two came back negative and are reported as such. The cell reads each record's own `answer` field, so nothing here is retyped from the README.

In [3]:
def record(name):
    with open(RUNS / f"{name}.json") as fh:
        return json.load(fh)

found = []
for name in ARM1:
    d = record(name)
    answer = d.get("answer", "")
    found.append({
        "experiment": d.get("experiment", name),
        "status": d.get("status", "?"),
        "corrections": sum(1 for k in d if k.startswith("correction")),
        "finding": (answer[:110] + "…") if len(answer) > 110 else answer,
    })

pd.set_option("display.max_colwidth", 120)
pd.DataFrame(found)

,experiment,status,corrections,finding
0,intake-validation,complete,3,"Yes, across three virus families. Every expected target ranks in the top eight with the pathogen held out, ide…"
1,holdout-ladder,complete,1,"The pipeline needs a relative sharing enough 5-mers, which in practice means one close enough to be conspecifi…"
2,identification-cascade,complete,0,"Yes, by two different mechanisms, and the split between them is the finding. Conserved enzymes are recoverable…"
3,cross-kingdom-intake,complete,0,"Yes, after replacing the candidate-selection step. The identification and ranking transfer unchanged; six-fram…"
4,macrocycle-surface-markers,complete,0,"Yes. Scoring the largest contiguous exposed hydrophobic patch, rather than pocket depth or enclosure, ranks th…"
5,surface-markers-in-the-ranker,complete,0,"No, in both forms tried, and the change is withdrawn. The markers are computed and reported; they are not scor…"
6,amenability-gate-v2,complete,0,It becomes honest rather than confident. Both defects are fixed and the fix to the second shows the gate canno…
7,coverage-channel,complete,0,No. It relocates the advantage from the count channel to the dossier-order channel at almost unchanged magnitu…


Two of these are **withdrawals**, and they are the reason to trust the other six.
`surface_in_ranker` tried the surface markers as a ranking feature in two forms, neither
helped, and the change was withdrawn: the markers are computed and reported, never scored.
`amenability_gate` turned a confident gate into an honest one by measuring the patch across
replicate structures of the same domain and finding it moves by up to 496 square angstroms
against a 600 threshold, which puts most real targets in a borderline band.

The `corrections` column counts correction fields written into the record *after* the fact.
`intake_validation` carries three, including the withdrawal of its own headline claim.

### 1.3 · Genome to ranked targets  ·  *to fill*

Reads `runs/target_ranks.csv` (16 rows, tracked). Planned: one table per pathogen showing
pipeline rank against the approved drug or antibody for that target, plus the rank-sum p-value
and, critically, the `verdict` column, which records Nipah as **not established
(underpowered)** rather than as a pass. Three virus families and one bacterium, with the
pathogen's own proteins removed from the reference before indexing.

### 1.4 · The identification cliff  ·  *to fill*

Reads `runs/holdout_ladder.json` and `runs/identification_cascade.json`. Planned: the ladder
for SARS-CoV-2 and Nipah showing that pushing the nearest curated relative further away makes
the pipeline **stop rather than degrade**, and the cascade result that past the cliff conserved
enzymes return through remote homology while family-specific surface proteins, the antibody
targets, have no distant relatives and are recovered from their own sequence as a class.

### 1.5 · The surface a macrocycle would bind  ·  *to fill*

Reads `runs/surface_markers.json` and `runs/amenability_gate.json`. Planned: a horizontal
interval chart of `patch_median` with `patch_range` as the bar, six targets, the 600 threshold
as a rule, coloured by gate verdict. The point it makes visually is that spike RBD outranks the
main protease, the reverse of a small-molecule druggability score and the right way round for
this project, and that the spread on spike RBD alone is 496 square angstroms.

---
## 2 · Variant ranking

### 2.1 · The surviving result

Arm 2's molecular work is blocked, so the available work was on public canonical-protein
datasets. It produced a result that turns out to be a statement about prediction rather than
about policy.

**Acquisition gain is governed by how densely the candidate pool covers the variable positions
the model has to learn from.** In plainer terms: a weak model fitted on 128 measurements
predicts held-out binding well enough to act on *only when the candidates share sequence
coordinates with the labelled ones*. Where they do not, correlation sits near 0.1 and nothing
downstream helps.

**Why this matters for library design.** The favourable regime is a randomised region of a few
positions over an alphabet, which is what an mRNA display library *is*. A weak predictor is
enough, but there must be one, and whether one can be fitted is knowable in advance.

#### One provenance note, which matters if anyone checks the arithmetic

The cell reads `mean_paired_gain` from the summary table the repo's own `summarize` command
writes. Those figures are **not** identical to the ones in the preregistration, and the reason
is benign: the preregistration was scored over the first 10 pools, while the summary table was
regenerated over every staged pool, up to 20 for some landscapes. The `pools` column makes the
difference visible. Quote the preregistered numbers when the claim is about the preregistered
prediction, and these when the claim is about all staged data.

In [4]:
GAIN = ARM2["method gain table"]

if not GAIN.exists():
    print(f"not on this machine: {GAIN.relative_to(ROOT)}\n"
          f"rebuild with:  python -m hpc_pilots summarize")
else:
    g = pd.read_csv(GAIN)
    at128 = g[(g.labels == 128) & (g.policy == "greedy")]

    # Variable positions per landscape, from PREREGISTRATION_COVERAGE.md. This is the
    # coverage axis and it is NOT in the gain table, so it is the one mapping typed by hand.
    # dlg4_pdz3_peptide is absent on purpose: the preregistration does not state a count
    # for it, and inventing one would put a made-up number in a results table.
    POSITIONS = {"grb2_sh3_peptide": 56, "rask_darpin_binding": 187,
                 "spg1_olson_igg_matched": 55, "yap1_ww_peptide": 34,
                 "spg1_olson_igg": 55, "spike_rbd_ace2": 2}

    show = at128.assign(variable_positions=at128.landscape.map(POSITIONS))[[
        "landscape", "protein", "variable_positions", "pool_size", "pools",
        "random_recall", "policy_recall", "mean_paired_gain", "pools_improved",
        "wilcoxon_p"]].sort_values("mean_paired_gain", ascending=False)

    print("greedy acquisition vs random, at a 128-label budget, paired per pool\n")
    display(show.reset_index(drop=True))
    print("\nmean_paired_gain is per-pool greedy recall minus random recall, then averaged.")
    print("A blank variable_positions means the preregistration states no count for it.")

greedy acquisition vs random, at a 128-label budget, paired per pool



,landscape,protein,variable_positions,pool_size,pools,random_recall,policy_recall,mean_paired_gain,pools_improved,wilcoxon_p
0,spg1_olson_igg_matched,Streptococcal protein G B1 domain,55.0,512,20,0.2228,0.5434,0.3206,18,0.0001
1,yap1_ww_peptide,YAP1 WW domain,34.0,512,18,0.2628,0.5064,0.2436,17,0.0003
2,grb2_sh3_peptide,GRB2 SH3 domain,56.0,512,20,0.2750,0.5115,0.2365,19,0.0001
3,rask_darpin_binding,KRAS,187.0,512,20,0.2558,0.4288,0.1731,14,0.0048
4,spg1_olson_igg,Streptococcal protein G B1 domain,55.0,4096,20,0.0280,0.1427,0.1146,14,0.0032
5,dlg4_pdz3_peptide,PSD-95 PDZ3 domain,NaN,256,5,0.6615,0.6769,0.0154,4,NaN
6,spike_rbd_ace2,SARS-CoV-2 spike receptor-binding domain,2.0,512,6,0.2699,0.2800,0.0101,2,NaN



mean_paired_gain is per-pool greedy recall minus random recall, then averaged.
A blank variable_positions means the preregistration states no count for it.


**The result is legible in that one table.** The four landscapes with a few dozen
variable positions over a 512-candidate pool gain between +0.11 and +0.32 against random, each
improving in 14 or more pools at a Wilcoxon p of 0.005 or below. The spike receptor-binding
domain pool, which varies only **2** positions, gains +0.010 and improves in 2 of 6 pools. Same
budget, same acquisition rule, same pool size. What differs is whether the candidates share
sequence coordinates with the labelled ones.

The PDZ3 row is the honest outlier and worth naming before anyone else does: its gain is +0.015
on 5 pools with no p-value, because at a 128-label budget on a 256-candidate pool the policy has
already labelled half the pool and random recall is 0.66. There is almost no headroom left to
win, which is a statement about that pool's geometry and not about acquisition.



### 2.2 · Tested by intervention  ·  *to fill*

Correlation would not have established the claim, so it was tested by intervening in both
directions with everything else held fixed. Planned: the two interventions side by side, read
from `ADJUDICATION_COMBINATORIAL_GAIN.md` and the staged runs.

| Intervention | Greedy gain | Model correlation |
| --- | --- | --- |
| Influenza H3, pools narrowed to 28 variable positions | +0.104 → **+0.250** | 0.10 → 0.63 |
| Same, widened to 55 / 150 / 566 positions | +0.159 / +0.019 / 0.000 | 0.42 / 0.18 / 0.10 |
| GB1 doubles re-encoded to share no feature levels | +0.242 → **−0.019** | 0.66 → 0.03 |

The first takes the pathogen case that appeared to fail and recovers the whole effect by
narrowing the pool. The second keeps the same variants, the same hidden scores and the same
combinatorial structure, destroys only the feature sharing, and removes the effect entirely.

It then held on two unrelated proteins against a prediction **written down and committed
first**. Predicted at least +0.15 improving in at least 7 of 10 pools; observed +0.254 and
+0.254, every criterion passed, permutation test over landscape labels p = 0.014.

### 2.3 · What Arm 2 is blocked on  ·  *to fill*

Planned: the null chemistry fields enumerated from the repo rather than asserted, so the
blockage is checkable. Every platform-supplied field is null: chemical identifiers, SMILES,
stereochemistry, reprogrammed codons, Rosetta parameters, rotamer libraries. All of it waits on
the SynX campaign export. The macrocycle chemistry is represented and tested but wired into no
experiment, and the physics-based non-canonical track is untouched.

That blockage is the critical path, not a line in a risk table. It is also why the arm above is
the one with results: when the molecular work could not proceed, decision-policy work on public
data could.

---
## 3 · What to say about this  ·  *to fill*

Planned: three claims worth defending and the one caveat that belongs beside them.

1. **Arm 1 selects by reachability**, which is a different and more useful question than
   relevance, and the dependency graph it exposes has a single bottleneck.
2. **Arm 1 has run on public data** across three virus families and one bacterium, and reports
   its two negative results and three corrections in the same records as its successes.
3. **Arm 2 has a condition for when acquisition works**, tested by intervention in both
   directions and preregistered before replication.
4. **The caveat**: none of it has touched a macrocycle yet. Arm 2 is blocked on the campaign
   export, and that is the critical path.